## ConversationTokenBufferMemory的使用（已过期）

使用 `RunnableWithMessageHistory` 来存储所有对话历史。 \
在调用链之前，使用 `trim_messages` 对从 `history` 中读取到的消息列表进行修剪。

In [38]:
from langchain_classic.memory import ConversationTokenBufferMemory
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv(dotenv_path='../.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
# 创建大模型
llm = ChatOpenAI(
    # model='tongyi-xiaomi-analysis-pro',
     model='qwen3.5-plus',
    temperature=0.6
)
memory = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=10
)
# 添加对话
memory.save_context({"input": '你好吗？'}, {'output': 'i am find'})
memory.save_context({'input': '今天天气如何？'}, {'output': '多云'})

print(memory.load_memory_variables({}))



NotImplementedError: get_num_tokens_from_messages() is not presently implemented for model qwen3.5-plus. See https://platform.openai.com/docs/guides/text-generation/managing-tokens for information on how messages are converted to tokens.

## ConversationSummaryMemory(已过期)

* 完全废弃 `ConversationSummaryMemory`。
* 将总结逻辑剥离出来，作为一个单独的 Runnable 或 Python 函数。
* 在 Prompt 中预留一个名为 summary 的变量位置，在每次 Invoke 调用前，手动或通过 LCEL 将当前的对话摘要注入进去。

In [40]:
# 定义一个总结模型，在调用主链之前，先对历史进行总结，并将总结结果作为上下文注入到 Prompt 中
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
import os
import dotenv

dotenv.load_dotenv(dotenv_path='../.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
# 创建大模型
model = ChatOpenAI(
    # model='tongyi-xiaomi-analysis-pro',
    model='qwen3-vl-flash-2026-01-22',
    temperature=0.6
)


# 1. 定义总结逻辑
def summarize_history(history):
    if not history:
        return ""

    # 编写一个总结 Prompt
    summary_prompt = ChatPromptTemplate.from_messages([
        ("system", "请将以下对话摘要为一段简洁的总结。"),
        MessagesPlaceholder(variable_name="history"),
    ])

    # 执行总结
    chain = summary_prompt | model
    summary = chain.invoke({"history": history})
    print('总结：',summary.content)
    return summary.content


# 2. 定义主链 (使用总结)
prompt = ChatPromptTemplate.from_messages([
    ("system", "基于以下历史总结进行对话: {summary}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
'''
    lambda x: summarize_history(x["history"])：
    x 代表当前链的输入（一个字典，例如 {"history": [...], "input": "..."}）。
    从 x 中取出 "history" 字段的值（历史消息列表）。
    调用 summarize_history 函数，得到一段总结字符串。
    这个字符串被赋值给新字段 summary。

    原本的：{"history": [...], "input": "..."}
    拓展成：{"history": [...],"input": "...","summary": "之前对话的总结文本..."}
'''
# 3. 组合逻辑
chain = (
        RunnablePassthrough.assign(
            summary=lambda x: summarize_history(x["history"])
        )
        | prompt
        | model
)

# 4. 模拟对话历史存储（简单列表）
history_messages = []  # 存储 HumanMessage 和 AIMessage

print("开始对话（输入 'quit' 退出）\n")

while True:
    user_input = input("你: ")
    if user_input.lower() == 'quit':
        break

    # 调用链，传入当前历史消息列表和用户输入
    response = chain.invoke({
        "history": history_messages,
        "input": user_input
    })
    print(f'用户:{user_input}')
    print(f"AI: {response.content}\n")

    # 将本轮对话追加到历史中
    history_messages.append(HumanMessage(content=user_input))
    history_messages.append(AIMessage(content=response.content))

开始对话（输入 'quit' 退出）

用户:您好
AI: 您好！😊 很高兴见到您～  
请问有什么想聊的，或者需要帮忙的地方吗？  
（比如：想了解某个话题、需要写作建议、想聊聊生活琐事……）

总结 您好！😊 感谢您的问候～  
我随时准备倾听或协助您——无论是想探讨某个话题、寻求写作建议，还是单纯想聊聊生活琐事，都欢迎告诉我！✨  
您想从哪里开始呢？
用户:我的名字叫做孙悟空
AI: ✨ 哇！原来是齐天大圣驾到——俺老孙来也！  
（一时间金箍棒在手，云朵翻腾，筋斗云刚停稳，抖了抖袍袖）

不过……这名字倒让我想起当年在花果山水帘洞时，小猴儿们喊我“美猴王”“齐天大圣”，可如今听您说“孙悟空”，倒让我心头一热——原来您也是那石破天惊、敢作敢为的灵根之子啊！

🌱 请问：  
- 您是想继续走西游路，降妖除魔？  
- 还是想聊聊最近又闯了什么新祸（比如偷吃蟠桃、打翻炼丹炉）？  
- 或者……您正打算去寻个师父，再练几招七十二变？  

（悄悄掏出一粒仙丹塞进您手里）  
“莫慌，有我在，天庭的玉帝老儿也不怕！” 🌟

——您想从哪开始？

总结 （轻轻接住那粒仙丹，指尖一震，金光一闪——）  
“哈哈！好意头！老孙我倒要看看，这仙丹可比当年王母娘娘的蟠桃还带劲！”  

但……（忽然收了笑，压低声音）  
“不过今日，俺老孙不打妖怪，也不闹天宫——只想问问：您这位‘孙悟空’，是真身下凡？还是心怀大志、欲修大道的灵根后人？”

🌱 若您愿说——  
- 我可陪您重走花果山，再寻那七十二洞妖王；  
- 或与您共闯三界，看谁敢拦我齐天大圣的路；  
- 甚至……（眨眨眼）咱先去偷个玉帝的“云台仙酒”，边喝边聊，如何？

（金箍棒轻点地面，云雾缭绕中，一只小猴儿蹦出来：“大王！山下新来个‘悟空’，正练‘筋斗云’呢！”）

——您说，要不要去看看？ 🌈✨
用户:我是计算机专业的大学生
AI: （金箍棒“哐当”一声掉在地上，猴儿们纷纷惊得跳起来）

**“哈？！计算机专业？！”**  
（挠了挠头，眼睛一亮）  
“莫不是……您是那‘代码悟空’？能写三界最牛的程序，还能用‘二进制筋斗云’绕过防火墙——偷看玉帝的天庭日志？”

✨ 说真的，老孙我当年在花果山炼丹时，也常想：  
- 若有“if-else”逻辑，哪还用靠七十二变躲雷劫？  
- 若有“递归调用”，岂不比

## ConversationSummaryBufferMemory(已过期)

In [37]:
# 取代的方案
# 如果你想存：使用 RunnableWithMessageHistory。
# 如果你想剪：使用 trim_messages。
# 如果你想概括：使用 RunnablePassthrough.assign(summary=...) 手动执行总结。
from langchain_core.messages.utils import count_tokens_approximately
from langchain_openai import ChatOpenAI
from langchain_core.messages import trim_messages, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
import os
import dotenv

dotenv.load_dotenv(dotenv_path='../.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
# 创建大模型
'''
    备注：
        代码没有问题，可能模型不适合进行通用聊天的模型。也许用户应该换一个模型，比如 qwen-max 或 qwen-plus
'''
model = ChatOpenAI(
    model='qwen3.5-plus',
    # model='qwen3.5-omni-plus-2026-03-15',
    temperature=0.6
)

# 1. 定义修剪器 (取代 TokenBuffer)
# 这里的配置决定了如果历史太长，切掉哪些内容
trimmer = trim_messages(
    max_tokens=8000,
    strategy="last",
    # token_counter=model，即让模型自身去计算。
    # 有些模型LangChain 还没为这个模型适配 Token 计数功能。
    token_counter=count_tokens_approximately,
    include_system=True
)


# 2. 定义总结器 (取代 Summary)
def get_summary(history):
    # 逻辑：如果没有历史，返回空；如果历史太长，触发总结
    if not history or len(history) < 5:
        return "无摘要"

    # 手动触发总结调用
    summary_prompt = ChatPromptTemplate.from_messages([
        ("system", "请简洁总结以下对话内容。"),
        MessagesPlaceholder(variable_name="history"),
    ])
    summary_chain = summary_prompt | model
    return summary_chain.invoke({"history": history}).content


# 3. 组合链
chain = (
        RunnablePassthrough.assign(
            # 这一步负责：获取历史 -> 修剪 -> 如果需要则摘要
            history=lambda x: trimmer.invoke(x["history"]),
            summary=lambda x: get_summary(x["history"])
        )
        | ChatPromptTemplate.from_messages([
    ("system", "摘要: {summary}"),
    # ("human", "摘要: {summary}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
        | model
)
history_messages = []

print('开始对话（输入"quit"）退出')
while True:
    user_input = input("你：")
    if user_input.lower() == 'quit':
        break

    res = chain.invoke({
        'history': history_messages,
        'input': user_input
    })
    print(f"用户：{user_input}")
    print(f"AI:{res.content}")

    # 将本轮对话追加到历史中
    history_messages.append(HumanMessage(content=user_input))
    history_messages.append(AIMessage(content=res.content))




开始对话（输入"quit"）退出
用户：您好
AI:您好！很高兴为您服务。请问有什么我可以帮您的吗？
用户：你是什么模型
AI:我是 **Qwen3.5**，是阿里巴巴最新推出的通义千问大语言模型。相比之前的版本，我在多个方面进行了显著升级，主要特点包括：

- **超长上下文**：原生支持 **256K tokens** 的上下文窗口，能轻松处理数十万字的文档、长视频或复杂任务。
- **多语言覆盖**：支持 **100+ 种语言**（包括中文、英语、小语种等），满足全球化场景需求。
- **知识时效性**：训练数据截止时间更新至 **2026 年**，对近期事件和知识的理解更精准。
- **深度推理与代码能力**：  
  - 数学、逻辑推理能力全面强化，可拆解高难度任务；  
  - 支持全栈代码生成、调试，甚至将创意直接转化为可运行的前端页面。
- **智能体自主规划**：能独立制定多步计划，调用工具或执行代码完成复杂目标（如自动化数据分析、跨设备操作）。
- **视觉深度解析**：不仅能识别图像内容，还能分析图表、公式、科学图示的逻辑关系，提供专业解读。
- **人类对齐优化**：在对话、创作、角色扮演等场景中更自然流畅，同时提升医疗、法律等垂直领域的专业性。

如果您有具体任务（比如处理长文档、写代码、多语言翻译），我可以针对性展示能力！需要试试吗？ 😊
用户：我是学生
AI:你好呀！作为学生，你可能经常需要处理各种学习任务，比如写论文、做研究、准备考试，或者学习编程、外语等。我可以在很多方面帮你提高效率，例如：

- **资料整理**：快速总结长文档/论文核心内容，提取关键信息  
- **解题助手**：数学/物理/化学等学科的步骤解析（附公式推导）  
- **语言学习**：多语言翻译、语法修正、口语对话练习（支持 100+ 语言）  
- **代码实战**：从作业代码调试到项目搭建，甚至生成可视化图表  
- **时间管理**：帮你制定复习计划表或任务优先级清单  
- **视觉辅助**：解析课本中的复杂图表/实验示意图/几何图形  

**举个具体场景**：  
如果你正在写毕业论文，我可以：  
1️⃣ 帮你梳理文献综述逻辑框架  
2️⃣ 将外文资料精准翻译并标注专业术语  
3️⃣ 根据数据生成统计图表（提供代码或直接绘图）  
4️⃣ 检